In [1]:
import pandas as pd
import numpy as np

In [7]:
base_path = r"D:\JNP\analytics 30days July\AB testing\data"

users = pd.read_csv(base_path + r"\users.csv")
orders = pd.read_csv(base_path + r"\orders.csv")
events = pd.read_csv(base_path + r"\events.csv")

events["event_date"] = pd.to_datetime(events["event_date"])
orders["order_date"] = pd.to_datetime(orders["order_date"])

print("users shape:", users.shape)
print("orders shape:", orders.shape)
print("events shape:", events.shape)

display(users.head())
display(orders.head())
display(events.head())

users shape: (1000, 4)
orders shape: (1500, 5)
events shape: (5000, 5)


,user_id,signup_date,country,device
0,1,2026-04-13,US,desktop
1,2,2026-06-29,China,mobile
2,3,2026-04-03,Canada,mobile
3,4,2026-01-15,US,desktop
4,5,2026-04-17,Mexico,desktop


,order_id,user_id,order_date,amount,status
0,1,983,2026-04-09,84.66,completed
1,2,729,2026-04-12,0.96,completed
2,3,504,2026-03-22,66.57,completed
3,4,456,2026-03-20,11.68,completed
4,5,45,2026-03-08,7.13,cancelled


,event_id,user_id,event_date,event_type,traffic_source
0,1,4,2026-04-21,view_product,paid_search
1,2,854,2026-03-23,add_to_cart,social
2,3,640,2026-04-04,view_product,social
3,4,986,2026-05-06,purchase,organic
4,5,992,2026-04-03,checkout,paid_search


In [8]:
users.columns

Index(['user_id', 'signup_date', 'country', 'device'], dtype='object')

In [9]:
user_first_source = (
    events
    .sort_values(["user_id", "event_date"])
    .groupby("user_id")
    .agg(first_traffic_source=("traffic_source", "first"))
    .reset_index()
)

user_first_source.head()

,user_id,first_traffic_source
0,1,paid_search
1,2,email
2,3,social
3,4,email
4,5,organic


In [10]:
np.random.seed(42)

# Build user-level experiment table
experiment_users = (
    users[["user_id", "device"]]
    .merge(user_first_source, on="user_id", how="left")
)

experiment_users["experiment_group"] = np.random.choice(
    ["control", "treatment"],
    size=len(experiment_users),
    p=[0.5, 0.5]
)

experiment_users.head()

,user_id,device,first_traffic_source,experiment_group
0,1,desktop,paid_search,control
1,2,mobile,email,treatment
2,3,mobile,social,treatment
3,4,desktop,email,treatment
4,5,desktop,organic,control


In [11]:
# Define purchase outcome at the user level.
# A user is counted as purchased if they had at least one purchase event.

purchase_users = (
    events[events["event_type"] == "purchase"]
    .groupby("user_id")
    .size()
    .reset_index(name="purchase_event_count")
)

purchase_users["purchased"] = 1

experiment_analysis = experiment_users.merge(
    purchase_users[["user_id", "purchased"]],
    on="user_id",
    how="left"
)

experiment_analysis["purchased"] = (
    experiment_analysis["purchased"]
    .fillna(0)
    .astype(int)
)

experiment_analysis.head()

,user_id,device,first_traffic_source,experiment_group,purchased
0,1,desktop,paid_search,control,1
1,2,mobile,email,treatment,0
2,3,mobile,social,treatment,0
3,4,desktop,email,treatment,0
4,5,desktop,organic,control,0


In [12]:
group_size = (
    experiment_analysis
    .groupby("experiment_group")
    .agg(users=("user_id", "nunique"))
    .reset_index()
)

group_size

,experiment_group,users
0,control,503
1,treatment,497


In [13]:
device_balance = (
    experiment_analysis
    .groupby(["experiment_group", "device"])
    .agg(users=("user_id", "nunique"))
    .reset_index()
)

device_balance["group_total_users"] = (
    device_balance
    .groupby("experiment_group")["users"]
    .transform("sum")
)

device_balance["share_within_group"] = (
    device_balance["users"] / device_balance["group_total_users"]
)

device_balance.sort_values(["device", "experiment_group"])

,experiment_group,device,users,group_total_users,share_within_group
0,control,desktop,183,503,0.363817
3,treatment,desktop,173,497,0.348089
1,control,mobile,296,503,0.588469
4,treatment,mobile,300,497,0.603622
2,control,tablet,24,503,0.047714
5,treatment,tablet,24,497,0.048290


In [14]:
source_balance = (
    experiment_analysis
    .groupby(["experiment_group", "first_traffic_source"])
    .agg(users=("user_id", "nunique"))
    .reset_index()
)

source_balance["group_total_users"] = (
    source_balance
    .groupby("experiment_group")["users"]
    .transform("sum")
)

source_balance["share_within_group"] = (
    source_balance["users"] / source_balance["group_total_users"]
)

source_balance.sort_values(["first_traffic_source", "experiment_group"])

,experiment_group,first_traffic_source,users,group_total_users,share_within_group
0,control,email,136,500,0.272000
4,treatment,email,144,495,0.290909
1,control,organic,117,500,0.234000
5,treatment,organic,113,495,0.228283
2,control,paid_search,117,500,0.234000
6,treatment,paid_search,123,495,0.248485
3,control,social,130,500,0.260000
7,treatment,social,115,495,0.232323


In [15]:
conversion_by_group = (
    experiment_analysis
    .groupby("experiment_group")
    .agg(
        users=("user_id", "nunique"),
        purchasers=("purchased", "sum")
    )
    .reset_index()
)

conversion_by_group["conversion_rate"] = (
    conversion_by_group["purchasers"] / conversion_by_group["users"]
)

conversion_by_group

,experiment_group,users,purchasers,conversion_rate
0,control,503,181,0.359841
1,treatment,497,166,0.334004


In [16]:
control_rate = conversion_by_group.loc[
    conversion_by_group["experiment_group"] == "control",
    "conversion_rate"
].iloc[0]

treatment_rate = conversion_by_group.loc[
    conversion_by_group["experiment_group"] == "treatment",
    "conversion_rate"
].iloc[0]

absolute_lift = treatment_rate - control_rate
relative_lift = absolute_lift / control_rate

print(f"Control conversion rate: {control_rate:.2%}")
print(f"Treatment conversion rate: {treatment_rate:.2%}")
print(f"Absolute lift: {absolute_lift:.2%}")
print(f"Relative lift: {relative_lift:.2%}")

Control conversion rate: 35.98%
Treatment conversion rate: 33.40%
Absolute lift: -2.58%
Relative lift: -7.18%


In [17]:
from statsmodels.stats.proportion import proportions_ztest

# Make sure the order is control, treatment for easier interpretation
conversion_for_test = conversion_by_group.set_index("experiment_group").loc[
    ["control", "treatment"]
]

successes = conversion_for_test["purchasers"].values
samples = conversion_for_test["users"].values

z_stat, p_value = proportions_ztest(
    count=successes,
    nobs=samples
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Z-statistic: 0.8582
P-value: 0.3908


In [18]:
conversion_by_source = (
    experiment_analysis
    .groupby(["first_traffic_source", "experiment_group"])
    .agg(
        users=("user_id", "nunique"),
        purchasers=("purchased", "sum")
    )
    .reset_index()
)

conversion_by_source["conversion_rate"] = (
    conversion_by_source["purchasers"] / conversion_by_source["users"]
)

conversion_by_source.sort_values(["first_traffic_source", "experiment_group"])

,first_traffic_source,experiment_group,users,purchasers,conversion_rate
0,email,control,136,53,0.389706
1,email,treatment,144,48,0.333333
2,organic,control,117,34,0.290598
3,organic,treatment,113,38,0.336283
4,paid_search,control,117,46,0.393162
5,paid_search,treatment,123,35,0.284553
6,social,control,130,48,0.369231
7,social,treatment,115,45,0.391304


## Experiment Readout

This notebook analyzes a simulated A/B test for an e-commerce checkout flow. Users were randomly assigned to control and treatment groups with a nearly balanced split: 503 users in control and 497 users in treatment.

The control group achieved a purchase conversion rate of 35.98%, while the treatment group achieved a conversion rate of 33.40%. This represents an absolute lift of -2.58 percentage points and a relative lift of -7.18%.

A two-proportion z-test produced a p-value of 0.3908. At the 5% significance level, the difference is not statistically significant. Therefore, there is not enough evidence to conclude that the treatment checkout flow improved purchase conversion.

Device and first-traffic-source balance checks suggest that the simulated random assignment is reasonably balanced across major user segments. Segment-level conversion patterns vary by first traffic source, but these results should be treated as exploratory because each segment has a smaller sample size and was not tested as a pre-planned hypothesis.

Because this notebook uses simulated experiment assignment and does not enforce an experiment exposure window, the result should be interpreted as an A/B testing workflow demonstration rather than a real product launch decision.